# Split a large TIFF into parts of 3 GB or less

This notebook:
- accepts an input `.tif` file
- estimates bytes per pixel from the raster metadata
- computes a safe tile height so each output file stays under the target size
- writes multiple GeoTIFF parts with preserved metadata

**Notes**
- Output files are written as horizontal strips (`part_001`, `part_002`, etc.)
- Compression is enabled by default to help keep files smaller
- A safety factor is used because actual file size can vary due to metadata and compression


In [1]:
from pathlib import Path
import math
import rasterio
from rasterio.windows import Window

# =========================
# User inputs
# =========================
INPUT_TIF = r"D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\mosaic.tif"
OUTPUT_DIR = r"D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\mosaic_2025"
MAX_SIZE_GB = 3.0

# Extra margin so outputs stay safely below the requested size
# Lower value = smaller chunks
SAFETY_FACTOR = 0.90

# Use compression for smaller outputs
COMPRESS = "LZW"   # options: "LZW", "DEFLATE", None
TILED = True
BIGTIFF = "IF_SAFER"

# Optional override: set to an integer row count if you want fixed-height chunks.
# If None, the notebook calculates a height automatically from the size limit.
ROWS_PER_CHUNK = None


In [2]:
def estimate_bytes_per_pixel(src):
    """
    Estimate uncompressed bytes per pixel across all bands.
    """
    return src.count * (src.dtypes[0].itemsize if hasattr(src.dtypes[0], 'itemsize') else __import__('numpy').dtype(src.dtypes[0]).itemsize)


def compute_rows_per_chunk(src, max_size_gb=3.0, safety_factor=0.90):
    """
    Compute a safe number of rows per output chunk so each file is likely
    to remain below the size limit.
    """
    max_bytes = max_size_gb * (1024 ** 3) * safety_factor
    bpp = estimate_bytes_per_pixel(src)
    bytes_per_row = src.width * bpp
    if bytes_per_row <= 0:
        raise ValueError("Could not estimate bytes per row.")

    rows = int(max_bytes // bytes_per_row)
    return max(1, rows)


def split_tif_by_size(
    input_tif,
    output_dir,
    max_size_gb=3.0,
    safety_factor=0.90,
    rows_per_chunk=None,
    compress="LZW",
    tiled=True,
    bigtiff="IF_SAFER",
):
    """
    Split a TIFF into multiple GeoTIFF files, each targeted to be <= max_size_gb.

    Parameters
    ----------
    input_tif : str or Path
        Path to source TIFF.
    output_dir : str or Path
        Destination folder for chunk TIFFs.
    max_size_gb : float
        Maximum target size per output file in GB.
    safety_factor : float
        Margin applied to the size limit.
    rows_per_chunk : int or None
        If provided, use this fixed number of rows per chunk.
        If None, compute automatically from raster metadata.
    compress : str or None
        Compression option for GeoTIFF.
    tiled : bool
        Whether to write tiled GeoTIFFs.
    bigtiff : str
        BIGTIFF creation option.
    """
    input_tif = Path(input_tif)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if not input_tif.exists():
        raise FileNotFoundError(f"Input TIFF not found: {input_tif}")

    created_files = []

    with rasterio.open(input_tif) as src:
        width = src.width
        height = src.height
        count = src.count

        if rows_per_chunk is None:
            rows_per_chunk = compute_rows_per_chunk(src, max_size_gb=max_size_gb, safety_factor=safety_factor)

        if rows_per_chunk < 1:
            raise ValueError("rows_per_chunk must be at least 1.")

        n_parts = math.ceil(height / rows_per_chunk)

        print(f"Input file       : {input_tif}")
        print(f"Raster size      : {width} x {height}")
        print(f"Band count       : {count}")
        print(f"Rows per chunk   : {rows_per_chunk}")
        print(f"Expected parts   : {n_parts}")

        base_profile = src.profile.copy()
        base_name = input_tif.stem

        for part_idx, row_start in enumerate(range(0, height, rows_per_chunk), start=1):
            chunk_height = min(rows_per_chunk, height - row_start)
            window = Window(col_off=0, row_off=row_start, width=width, height=chunk_height)
            transform = src.window_transform(window)

            data = src.read(window=window)

            profile = base_profile.copy()
            profile.update({
                "height": chunk_height,
                "width": width,
                "transform": transform,
                "driver": "GTiff",
                "BIGTIFF": bigtiff,
            })

            if compress is not None:
                profile["compress"] = compress
            if tiled:
                profile["tiled"] = True

            out_path = output_dir / f"{base_name}_part_{part_idx:03d}.tif"

            with rasterio.open(out_path, "w", **profile) as dst:
                dst.write(data)

            size_gb = out_path.stat().st_size / (1024 ** 3)
            print(f"Created: {out_path.name}  |  {size_gb:.3f} GB")
            created_files.append(out_path)

    return created_files


In [3]:
created = split_tif_by_size(
    input_tif=INPUT_TIF,
    output_dir=OUTPUT_DIR,
    max_size_gb=MAX_SIZE_GB,
    safety_factor=SAFETY_FACTOR,
    rows_per_chunk=ROWS_PER_CHUNK,
    compress=COMPRESS,
    tiled=TILED,
    bigtiff=BIGTIFF,
)

print("\nDone.")
print(f"Total files created: {len(created)}")


Input file       : D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\mosaic.tif
Raster size      : 207575 x 133247
Band count       : 4
Rows per chunk   : 3491
Expected parts   : 39
Created: mosaic_part_001.tif  |  0.156 GB
Created: mosaic_part_002.tif  |  0.531 GB
Created: mosaic_part_003.tif  |  0.556 GB
Created: mosaic_part_004.tif  |  0.754 GB
Created: mosaic_part_005.tif  |  0.864 GB
Created: mosaic_part_006.tif  |  1.130 GB
Created: mosaic_part_007.tif  |  1.095 GB
Created: mosaic_part_008.tif  |  1.158 GB
Created: mosaic_part_009.tif  |  1.187 GB
Created: mosaic_part_010.tif  |  1.291 GB
Created: mosaic_part_011.tif  |  1.466 GB
Created: mosaic_part_012.tif  |  1.519 GB
Created: mosaic_part_013.tif  |  1.574 GB
Created: mosaic_part_014.tif  |  1.307 GB
Created: mosaic_part_015.tif  |  1.400 GB
Created: mosaic_part_016.tif  |  1.512 GB
Created: mosaic_part_017.tif  |  1.607 GB
Created: mosaic_part_018.tif  |  1.423 GB
Created: mosaic_part_019.tif  

## If any output still goes above 3 GB

Reduce the chunk size by either:

1. lowering `SAFETY_FACTOR` from `0.90` to `0.80`, or
2. setting `ROWS_PER_CHUNK` manually to a smaller value.

Example:

```python
ROWS_PER_CHUNK = 5000
```

That gives you direct control over the split size.
